In [1]:
import pandas as pd
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Train DataFrame head:")
display(train_df.head())

print("\nTest DataFrame head:")
display(test_df.head())

Train DataFrame head:


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A



Test DataFrame head:


,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


### Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [2]:
ac = train_df['answer'].value_counts()

print("Frequency distribution of 'answer' column:")
display(ac)

most_frequent_option = ac.idxmax()
most_frequent_count = ac.max()

least_frequent_option = ac.idxmin()
least_frequent_count = ac.min()

sum_of_occurrences = most_frequent_count + least_frequent_count

print(f"\nMost frequent option: {most_frequent_option} (Count: {most_frequent_count})")
print(f"Least frequent option: {least_frequent_option} (Count: {least_frequent_count})")
print(f"Sum of occurrences of the most frequent and least frequent options: {sum_of_occurrences}")

Frequency distribution of 'answer' column:


answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


Most frequent option: B (Count: 490)
Least frequent option: E (Count: 324)
Sum of occurrences of the most frequent and least frequent options: 814


### After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [3]:
import string

cpm = train_df['prompt'].str.lower()

translation_table = str.maketrans('', '', string.punctuation)
cpm = cpm.apply(lambda x: x.translate(translation_table))

all_words = []
for prompt in cpm:
    all_words.extend(prompt.split())

vocabulary_size = len(set(all_words))

print(f"Total number of unique words (vocabulary size) in the cleaned 'prompt' column: {vocabulary_size}")

Total number of unique words (vocabulary size) in the cleaned 'prompt' column: 859


### Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  


In [4]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

prompt_row_1 = cpm.iloc[0]

words_row_1 = prompt_row_1.split()

filtered_words_row_1 = [word for word in words_row_1 if word not in ENGLISH_STOP_WORDS]

num_words_after_filtering = len(filtered_words_row_1)

print(f"The prompt for Row ID 1 after filtering out stop words: {filtered_words_row_1}")
print(f"Number of words left in the prompt for Row ID 1 after filtering: {num_words_after_filtering}")

The prompt for Row ID 1 after filtering out stop words: ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
Number of words left in the prompt for Row ID 1 after filtering: 13


### Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer


combined_text = (
    train_df['prompt'].astype(str) + " " +
    train_df['A'].astype(str) + " " +
    train_df['B'].astype(str) + " " +
    train_df['C'].astype(str) + " " +
    train_df['D'].astype(str) + " " +
    train_df['E'].astype(str)
)

vectorizer = TfidfVectorizer(stop_words='english')

vectorizer.fit(combined_text)

vocabulary_size_tfidf = len(vectorizer.vocabulary_)

print(f"Total number of feature columns (vocabulary size) generated by TfidfVectorizer: {vocabulary_size_tfidf}")

Total number of feature columns (vocabulary size) generated by TfidfVectorizer: 2762


### Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  



In [6]:
from sklearn.metrics.pairwise import cosine_similarity

prompt_row_1 = train_df.loc[0, 'prompt']
option_A_row_1 = train_df.loc[0, 'A']

prompt_vector = vectorizer.transform([prompt_row_1])
option_A_vector = vectorizer.transform([option_A_row_1])

cosine_sim = cosine_similarity(prompt_vector, option_A_vector)[0][0]

rounded_cosine_sim = round(cosine_sim, 4)

print(f"Cosine similarity between prompt and option A for Row ID 1: {rounded_cosine_sim}")

Cosine similarity between prompt and option A for Row ID 1: 0.272


### Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

cp = 0
n = len(train_df)

option_cols = ['A', 'B', 'C', 'D', 'E']

for index, row in train_df.iterrows():
    prompt = row['prompt']
    correct_answer = row['answer']

    prompt_vector = vectorizer.transform([prompt])

    max_similarity = -1
    predicted_option = None

    for option_char in option_cols:
        option_text = row[option_char]
        option_vector = vectorizer.transform([option_text])

        sim = cosine_similarity(prompt_vector, option_vector)[0][0]

        if sim > max_similarity:
            max_similarity = sim
            predicted_option = option_char

    if predicted_option == correct_answer:
        cp += 1

accuracy_percentage = (cp / n) * 100

print(f"Total rows: {n}")
print(f"Correct pred based on highest cosine similarity: {cp}")
print(f"Percentage of instances where highest cosine similarity matches the correct answer: {accuracy_percentage:.2f}%")

Total rows: 2000
Correct pred based on highest cosine similarity: 271
Percentage of instances where highest cosine similarity matches the correct answer: 13.55%


### If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  

In [8]:
def calculate_ap_at_k(gt, pred, k):
    """
    Calculates Average Precision at k (AP@k) for a single query.

    Args:
        gt (str): The single correct answer.
        pred (list): A list of predicted answers, ordered by relevance.
        k (int): The maximum number of pred to consider.

    Returns:
        float: The AP@k score for the query.
    """
    if not gt:
        return 0.0

    relevant_found = 0
    sum_precisions = 0.0

    for i, pred in enumerate(pred[:k]):
        if pred == gt:
            relevant_found += 1
            precision_at_rank = relevant_found / (i + 1)
            sum_precisions += precision_at_rank

    return sum_precisions if relevant_found > 0 else 0.0

ground_truth_example = 'C'
predictions_example = ['C', 'A', 'B']
k_example = 3

map_at_3_score = calculate_ap_at_k(ground_truth_example, predictions_example, k_example)
print(f"Ground truth: {ground_truth_example}")
print(f"Predictions: {predictions_example}")
print(f"AP@{k_example} (MAP@{k_example} for a single query): {map_at_3_score}")

Ground truth: C
Predictions: ['C', 'A', 'B']
AP@3 (MAP@3 for a single query): 1.0


### If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E? 

In [9]:
def calculate_ap_at_k(gt, pred, k):
    """
    Calculates Average Precision at k (AP@k) for a single query.

    Args:
        gt (str): The single correct answer.
        pred (list): A list of predicted answers, ordered by relevance.
        k (int): The maximum number of pred to consider.

    Returns:
        float: The AP@k score for the query.
    """
    if not gt:
        return 0.0

    relevant_found = 0
    sum_precisions = 0.0

    for i, pred in enumerate(pred[:k]):
        if pred == gt:
            relevant_found += 1
            precision_at_rank = relevant_found / (i + 1)
            sum_precisions += precision_at_rank

    return sum_precisions if relevant_found > 0 else 0.0
ground_truth_example = 'B'
predictions_example = ['D', 'B', 'E']
k_example = 3

map_at_3_score = calculate_ap_at_k(ground_truth_example, predictions_example, k_example)
print(f"Ground truth: {ground_truth_example}")
print(f"Predictions: {predictions_example}")
print(f"AP@{k_example} (MAP@{k_example} for a single query): {map_at_3_score}")

Ground truth: B
Predictions: ['D', 'B', 'E']
AP@3 (MAP@3 for a single query): 0.5


### The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [10]:
top_3_answers = ac.head(3).index.tolist()

print(f"Top 3 most frequent answers: {top_3_answers}")

baseline_predictions = top_3_answers

ap_at_3_scores = []

for index, row in train_df.iterrows():
    gt = row['answer']

    ap_score = calculate_ap_at_k(gt, baseline_predictions, k=3)
    ap_at_3_scores.append(ap_score)

overall_map_at_3 = sum(ap_at_3_scores) / len(ap_at_3_scores)

print(f"\nOverall MAP@3 score for the Majority Class Baseline on train.csv: {overall_map_at_3:.4f}")

Top 3 most frequent answers: ['B', 'C', 'A']

Overall MAP@3 score for the Majority Class Baseline on train.csv: 0.4213


### The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

tfidf_ap_at_3_scores = []

option_cols = ['A', 'B', 'C', 'D', 'E']

for index, row in train_df.iterrows():
    prompt = row['prompt']
    gt = row['answer']

    prompt_vector = vectorizer.transform([prompt])

    similarities = []
    for option_char in option_cols:
        option_text = row[option_char]
        option_vector = vectorizer.transform([option_text])

        sim = cosine_similarity(prompt_vector, option_vector)[0][0]
        similarities.append((sim, option_char))

    similarities.sort(key=lambda x: x[0], reverse=True)

    top_3_predictions = [s[1] for s in similarities[:3]]

    ap_score = calculate_ap_at_k(gt, top_3_predictions, k=3)
    tfidf_ap_at_3_scores.append(ap_score)

overall_tfidf_map_at_3 = sum(tfidf_ap_at_3_scores) / len(tfidf_ap_at_3_scores)

print(f"\nOverall MAP@3 score for the TF-IDF Pipeline on train.csv: {overall_tfidf_map_at_3:.4f}")


Overall MAP@3 score for the TF-IDF Pipeline on train.csv: 0.2962
